In [ ]:
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
from ts2.utils.tailwind import TC
from tqdm import tqdm

In [ ]:
out_dir: str = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ns/utils/histreg/results/status"
csv_files: list[str] = [
    "accepted.csv",
    "new_accepted.csv",
    "no_he.csv",
    "one_section.csv",
]

frames: list[pd.DataFrame] = [
    pd.read_csv(os.path.join(out_dir, csv_file))
    for csv_file in csv_files
]
merged: pd.DataFrame = pd.concat(frames, ignore_index=True, sort=False)
merged

In [ ]:
merged["which"].drop_duplicates()

In [ ]:
matched_su_mxa_fname: str = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ns/utils/scanning/out/matched_su_mxa.csv"
matched_su_mxa: pd.DataFrame = pd.read_csv(matched_su_mxa_fname, dtype=str)
blocks = (matched_su_mxa["UM Accession"] + "." + matched_su_mxa["Block"]).drop_duplicates()


In [ ]:
merged_block_which: pd.DataFrame = merged[["block", "which"]].drop_duplicates()

block_which_counts: pd.Series = merged_block_which.groupby("block")["which"].nunique()
multiple_blocks: pd.Index = block_which_counts[block_which_counts > 1].index
multiple_block_which: pd.DataFrame = merged_block_which[
    merged_block_which["block"].isin(multiple_blocks)
].sort_values(["block", "which"]).reset_index(drop=True)
display(multiple_block_which)

assert len(multiple_block_which) == 0


In [ ]:

which_by_block: pd.Series = merged_block_which.groupby("block")["which"].first()
all_blocks: pd.Series = pd.concat(
    [pd.Series(blocks, name="block"), merged_block_which["block"]],
    ignore_index=True,
).drop_duplicates()
block_chart: pd.DataFrame = pd.DataFrame({"block": all_blocks}).merge(
    which_by_block.rename("which"),
    left_on="block",
    right_index=True,
    how="left",
)
matched_blocks: pd.Series = pd.Series(blocks, name="block")
intraop_frozen_blocks: set[str] = set(
    matched_blocks[
        matched_blocks.str.contains(r"\.[A-Za-z]+[0-9]+x$", regex=True, na=False)
    ]
)
block_chart.loc[
    block_chart["which"].isna() & block_chart["block"].isin(intraop_frozen_blocks),
    "which",
] = "intraop_frozen"
block_chart["which"] = block_chart["which"].fillna("todo")

which_order: list[str] = [
    "no_he",
    "intraop_frozen",
    "one_section_only",
    "block_align_crop_affine",
    "block_align_crop_affine_ransac",
    "block_align_crop_rigid_ransac",
    "block_align_manual",
    "todo",
]
which_colors: dict[str, str] = {
    "no_he": TC.red[6],
    "intraop_frozen": TC.violet[5],
    "one_section_only": TC.fuchsia[7],
    "block_align_crop_affine": TC.lime[5],
    "block_align_crop_affine_ransac": TC.green[6],
    "block_align_crop_rigid_ransac": TC.emerald[7],
    "block_align_manual": TC.teal[6],
    "todo": TC.sky[6],
}
unknown_which: set[str] = set(block_chart["which"].dropna()) - set(which_order)
if unknown_which:
    raise ValueError(f"Missing order for which values: {sorted(unknown_which)}")

block_counts: pd.Series = block_chart["which"].value_counts().reindex(
    which_order,
    fill_value=0,
)
plot_counts: pd.Series = block_counts[block_counts > 0]
slice_labels: list[str] = [
    f"{which} ({int(count)})"
    for which, count in plot_counts.items()
]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    plot_counts,
    labels=slice_labels,
    colors=[which_colors[which] for which in plot_counts.index],
    startangle=90,
    counterclock=True,
    labeldistance=1.08,
    wedgeprops={"width": 0.1, "edgecolor": "none", "linewidth": 0, "alpha": 1.0},
)

ax.set_title(f"Blocks by alignment outcome (n={int(block_counts.sum())})")
ax.set(aspect="equal")
plt.show()

In [ ]:
histreg_dir: str = "/nfs/turbo/umms-tocho-snr/exp/chengjia/neuroslides/histreg"
algorithms: list[str] = [
    "block_align_crop_affine",
    "block_align_crop_affine_ransac",
    "block_align_crop_rigid_ransac",
    "block_align_manual",
]
required_suffixes: list[str] = [
    "_align_viz.pkl",
    "_align.pkl",
    "_im_align.pdf",
    "_mask_align.pdf",
]

todo_blocks: pd.Series = block_chart.loc[
    block_chart["which"] == "todo",
    "block",
].drop_duplicates().sort_values().reset_index(drop=True)

processed_by_algorithm: dict[str, set[str]] = {}
for algorithm in algorithms:
    algorithm_dir: str = os.path.join(histreg_dir, algorithm)
    filenames: set[str] = set(os.listdir(algorithm_dir))
    candidate_blocks: set[str] = {
        filename[: -len(suffix)]
        for suffix in required_suffixes
        for filename in filenames
        if filename.endswith(suffix)
    }
    processed_by_algorithm[algorithm] = {
        block
        for block in candidate_blocks
        if all(f"{block}{suffix}" in filenames for suffix in required_suffixes)
    }

todo_processed: pd.DataFrame = pd.DataFrame({"block": todo_blocks})
for algorithm in algorithms:
    todo_processed[algorithm] = todo_processed["block"].isin(
        processed_by_algorithm[algorithm]
    )

todo_processed

In [ ]:
todo_processed.loc[todo_processed["block_align_crop_affine_ransac"], "block"]#.to_csv("toreview_2605a.csv", index=False)

In [ ]:
todo_processed.loc[todo_processed[algorithms].any(axis=1)==False, "block"]#.to_csv("toalign_2605b.csv", index=False)